In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import (
    PROJECT_ROOT,
    DATA_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    WEATHER_DIR,
    SOIL_DIR,
    RAW_DIR,
    OUTPUT_DIR,
    PROCESSED_DATASET,
    WEATHER_FEATHER,
    MERGED_DATA_DIR,
    interim_csb_path,
)


In [ ]:
import pandas as pd
import os

# Legacy weather CSV → feather conversion (source in processed/niss_legacy)
input_file = PROCESSED_DIR / "niss_legacy" / "Weather.csv"
output_file = WEATHER_FEATHER

try:
    df = pd.read_csv(input_file)
    print(f"Successfully loaded data from: {input_file}")
    print(df.info())
    df.to_feather(output_file)
    print(f"Successfully saved Feather file to: {output_file}")
except FileNotFoundError:
    print(f"Error: The file was not found at {input_file}. Check the path.")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
# Import required library
import pandas as pd
from pathlib import Path

# Load CSB dataset (Crop Sequence Boundaries)
csb = pd.read_feather(interim_csb_path("20172024"))

# Load annual county-level weather dataset
weather = pd.read_feather(WEATHER_FEATHER)


In [ ]:
# Define a helper function to normalize county names
def clean_county(s):
    return (
        s.astype(str)
         .str.lower()
         .str.strip()
         .str.replace(" county", "", regex=False)
    )

# Apply normalization to county names in both datasets
csb["county"] = clean_county(csb["CNTY"])
weather["county"] = clean_county(weather["county_name"])


In [ ]:
# Identify CDL crop columns (one column per year)
cdl_cols = [f"CDL{y}" for y in range(2017, 2025)]

# Optional sanity check: confirm all columns exist
missing_cols = set(cdl_cols) - set(csb.columns)
print("Missing CDL columns:", missing_cols)


In [ ]:
# Reshape CSB data: convert yearly CDL columns into long format
# Each row becomes: county × year × field × crop
csb_long = csb.melt(
    id_vars=["county", "CSBACRES"],
    value_vars=cdl_cols,
    var_name="year",
    value_name="crop_code"
)

# Quick sanity check
csb_long.head()


In [ ]:
# STEP 5:
# Convert the 'year' column from strings like 'CDL2017'
# into a numeric year variable (e.g., 2017) for aggregation and merging

csb_long["year"] = csb_long["year"].str[-4:].astype(int)

# Sanity check
csb_long[["year"]].drop_duplicates().sort_values("year")


In [ ]:
# STEP 6:
# Aggregate CSB data to county–year–crop level by summing field acres
# This produces total crop acreage for each county in each year

csb_agg = (
    csb_long
    .groupby(["county", "year", "crop_code"], as_index=False)
    .agg(total_acres=("CSBACRES", "sum"))
)

# Sanity check: inspect the aggregated output
csb_agg.head()


In [ ]:
# Keep only years that exist in CSB (2017–2024)
weather_annual = weather_annual[
    weather_annual["year"].between(2017, 2024)
]

weather_annual["year"].unique()

In [ ]:
# STEP 7:
# Align year column naming between datasets and merge
# Keep ALL weather columns, plus crop_code and total_acres from CSB


# Keep only years that exist in CSB (2017–2024)
weather_annual = weather_annual[
    weather_annual["year"].between(2017, 2024)
]

weather_annual["year"].unique()


# Drop duplicate county column to avoid confusion
weather_annual = weather_annual.drop(columns=["county_name"])



# Merge weather with CSB acreage data
final_merged = weather_annual.merge(
    csb_agg[["county", "year", "crop_code", "total_acres"]],
    on=["county", "year"],
    how="left"
)

# Sanity checks
final_merged.head(), final_merged.shape


In [ ]:
# STEP 10:
# Export the final merged dataset to CSV format
# Save it to the predefined BASE_PATH

# Inspect merged dataset size
print("Final dataset shape:", final_merged.shape)

# Preview first few rows
final_merged.head() 

output_path = PROCESSED_DIR / "csb_weather_merged.csv"

final_merged.to_csv(output_path, index=False)

print(f"Dataset successfully exported to: {output_path}")


# crosswalk_build.py

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd


#  Load data

csb = pd.read_csv("ny_csb_20172024.csv")
soil = pd.read_csv("soil_features_NY.csv")
counties = gpd.read_file("tl_2025_us_county.shp")
counties.head()

#print("CSB shape:", csb.shape)
#print("Soil shape:", soil.shape)

#print(csb.head())
#print(soil.head())
#print(counties.head())

#Filter US counties to New York + create a county key
ny_counties = counties[counties["STATEFP"] == "36"].copy()
ny_counties["CNTYFIPS"] = ny_counties["COUNTYFP"].astype(str).str.zfill(3)
ny_counties["STATEFIPS"] = ny_counties["STATEFP"].astype(str).str.zfill(2)
ny_counties["county_key"] = ny_counties["STATEFIPS"] + ny_counties["CNTYFIPS"]

##print(ny_counties[["county_key","NAME"]].head())

soil_poly = gpd.read_file("gSSURGO_NY.gdb", layer="MUPOLYGON")
#print(soil_poly.columns)
#print(soil_poly[["MUKEY"]].head())

# ============================================================
# Step 1. Ensure both layers share the same CRS
# ============================================================
soil_poly = soil_poly.to_crs(ny_counties.crs)

soil_poly = soil_poly[["MUKEY", "geometry"]].copy()
ny_counties = ny_counties[["county_key", "geometry"]].copy()

# ============================================================
# Step 2. Spatial join to generate candidate soil–county pairs
# ============================================================
pairs = gpd.sjoin(
    soil_poly,
    ny_counties,
    how="inner",
    predicate="intersects"
).reset_index(drop=True)

# IMPORTANT: after sjoin, make sure county_key exists
# (it should come from ny_counties)
print("Columns after sjoin:", pairs.columns.tolist())

if "county_key" not in pairs.columns:
    # If county_key is missing, it means the join did not carry it over
    # (rare, but can happen if ny_counties columns were not kept correctly)
    raise ValueError("county_key is not present after sjoin. Please check ny_counties columns.")

print("Candidate soil–county pairs:", pairs.shape)

# ============================================================
# Step 3. Reproject to a projected CRS BEFORE computing areas
# ------------------------------------------------------------
# Area computations in a geographic CRS (degrees) are invalid.
# EPSG:3857 uses meters and is sufficient for area weights.
# ============================================================
pairs = pairs.to_crs(epsg=3857)
ny_counties_m = ny_counties.to_crs(epsg=3857)

# Attach county geometry for exact intersection
pairs = pairs.merge(
    ny_counties_m[["county_key", "geometry"]].rename(columns={"geometry": "county_geom"}),
    on="county_key",
    how="left"
)

# Compute exact intersection geometry
pairs["intersect_geom"] = pairs.geometry.intersection(pairs["county_geom"])

# Compute intersection area (square meters)
pairs["intersect_area"] = pairs["intersect_geom"].area

# Keep only positive-area overlaps
pairs = pairs[pairs["intersect_area"] > 0].copy()
print("Valid intersecting pairs:", pairs.shape)

# ============================================================
# Step 4. Compute area fractions within each MUKEY
# ============================================================
pairs["area_frac"] = (
    pairs["intersect_area"] /
    pairs.groupby("MUKEY")["intersect_area"].transform("sum")
)

# Final crosswalk table
crosswalk = pairs[["MUKEY", "county_key", "area_frac"]].copy()

print(crosswalk.head())
print("Crosswalk shape:", crosswalk.shape)

crosswalk.to_csv("mukey_county_crosswalk.csv", index=False)
print("Saved mukey_county_crosswalk.csv")



# postprocess_county_merge.py

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Load precomputed crosswalk (from spatial steps)
# ============================================================
# ============================================================
# Step 5. Soil: county-level aggregation (area-weighted)
# ============================================================
crosswalk = pd.read_csv("mukey_county_crosswalk.csv")
soil_attr = pd.read_csv("soil_features_NY.csv")

# Treat MUKEY as an ID (string) to avoid dtype mismatch
crosswalk["MUKEY"] = crosswalk["MUKEY"].astype(str)
soil_attr["MUKEY"] = soil_attr["MUKEY"].astype(str)

# Keep only the columns we need to reduce memory
soil_num_cols = soil_attr.select_dtypes(include=[np.number]).columns.tolist()
soil_num_cols = [c for c in soil_num_cols if c != "MUKEY"]
soil_attr_small = soil_attr[["MUKEY"] + soil_num_cols].copy()

# Merge: MUKEY–county pairs + soil variables
soil_join = crosswalk.merge(soil_attr_small, on="MUKEY", how="left")

# ------------------------------------------------------------
# Vectorized area-weighted aggregation (NO groupby-apply)
# ------------------------------------------------------------
# 1) Multiply each soil variable by area_frac
w = pd.to_numeric(soil_join["area_frac"], errors="coerce").astype("float64").fillna(0.0)

for col in soil_num_cols:
    soil_join[col] = pd.to_numeric(soil_join[col], errors="coerce").astype("float64")
    soil_join[col] = soil_join[col] * w

# 2) Sum weighted values by county_key
soil_county_sum = soil_join.groupby("county_key", as_index=False)[soil_num_cols].sum()

# 3) Also compute sum of weights by county (for weighted mean)
w_sum = soil_join.groupby("county_key", as_index=False)["area_frac"].sum().rename(columns={"area_frac": "w_sum"})

# 4) Divide weighted sums by weight sums → weighted mean
soil_county = soil_county_sum.merge(w_sum, on="county_key", how="left")
for col in soil_num_cols:
    soil_county[col] = soil_county[col] / soil_county["w_sum"]

soil_county = soil_county.drop(columns=["w_sum"])

print("Soil county-level shape:", soil_county.shape)
soil_county.to_csv("soil_county_aggregated.csv", index=False)
print("Saved soil_county_aggregated.csv")



# ============================================================
# Step 6. CSB: county-level aggregation
# ============================================================
csb = pd.read_csv("ny_csb_20172024.csv")

csb["STATEFIPS"] = csb["STATEFIPS"].astype(str).str.zfill(2)
csb["CNTYFIPS"] = csb["CNTYFIPS"].astype(str).str.zfill(3)
csb["county_key"] = csb["STATEFIPS"] + csb["CNTYFIPS"]

csb_num_cols = csb.select_dtypes(include=[np.number]).columns.tolist()
csb_num_cols = [c for c in csb_num_cols if c != "CSBID"]

csb_county = csb.groupby("county_key", as_index=False)[csb_num_cols].mean()

# Force one row per county_key (safety)
csb_county_1 = csb_county.groupby("county_key", as_index=False).mean(numeric_only=True)

print("CSB county-level:", csb_county_1.shape)

# ============================================================
# Step 7. Merge (safe one-to-one)
# ============================================================

# Ensure county_key has the same dtype in both tables
csb_county_1["county_key"] = csb_county_1["county_key"].astype(str).str.zfill(5)
soil_county["county_key"] = soil_county["county_key"].astype(str).str.zfill(5)

final = csb_county_1.merge(soil_county, on="county_key", how="left", indicator=True)

print("Merge indicator counts:")
print(final["_merge"].value_counts())

final.to_csv("csb_soil_county_merged.csv", index=False)
print("Saved csb_soil_county_merged.csv")


# eda_county_analysis.py

In [ ]:
import pandas as pd
import numpy as np

# Load final merged county-level dataset
final = pd.read_csv("csb_soil_county_merged.csv")

print("Final dataset shape:", final.shape)
print(final.head())

# Missing rate by column
na_rate = final.isna().mean().sort_values(ascending=False)
print(na_rate)

soil_cols = [c for c in final.columns if "gSSURGO" in c]

soil_desc = final[soil_cols].describe().T
print(soil_desc)

# CSB outcome variables (you can adjust later)
csb_vars = [
    "CSBYEARS",
    "CSBACRES",
    "CDL2017", "CDL2018", "CDL2019",
    "CDL2020", "CDL2021", "CDL2022",
    "CDL2023", "CDL2024"
]

# Soil variables (all gSSURGO-derived)
soil_vars = [c for c in final.columns if "gSSURGO" in c]

print("Number of soil variables:", len(soil_vars))

# Subset and compute correlation
corr = final[soil_vars + csb_vars].corr()

# Only keep soil vs CSB block
soil_csb_corr = corr.loc[soil_vars, csb_vars]

top_corr = (
    soil_csb_corr["CSBACRES"]
    .sort_values(key=np.abs, ascending=False)
    .head(10)
)

print("Top soil variables correlated with CSBACRES:")
print(top_corr)
